## 	Cyclistic bike-share analysis case study Data Cleaning and Transformation

In [1]:
import pandas as pd
import glob
import os

### Step 1: Define the path to your dataset folder

In [20]:
folder_path = r"C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset"

### Step 2 : Load all CSV files from the folder

In [21]:
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

dfs = []
for f in all_files:
    try:
        df = pd.read_csv(f)
        dfs.append(df)
        print(f"Loaded {f} with shape {df.shape}")
    except Exception as e:
        print(f"Error reading {f}: {e}")

Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\APR_2025.csv with shape (371341, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\AUG_2025.csv with shape (790177, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\DEC_2025.csv with shape (140534, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\FEB_2025.csv with shape (151880, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\JAN_2025.csv with shape (138689, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\JUL_2025.csv with shape (763432, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE DATA ANALYTICS\MOD 9\Case study 1\Dataset\JUN_2025.csv with shape (678904, 13)
Loaded C:\Users\Gurunath Chavan\Desktop\LEARNING\GOOGLE

### Step 3: Merge into one DataFrame

In [23]:
data = pd.concat(dfs, ignore_index=True)

### Step 4: Standardize column names


In [24]:
data.columns = data.columns.str.lower().str.strip()

### Step 5: Drop missing values in critical fields

In [25]:
data = data.dropna(subset=["ride_id", "started_at", "ended_at", "member_casual"])

### Step 6:  Remove duplicates

In [26]:
data = data.drop_duplicates(subset="ride_id")

### Step 7: Convert timestamps

In [27]:
data["started_at"] = pd.to_datetime(data["started_at"])
data["ended_at"] = pd.to_datetime(data["ended_at"])

### Step 8: Ensure ride_id is string

In [28]:
data["ride_id"] = data["ride_id"].astype(str)

### Step 9: Normalize rider type labels

In [29]:
data["member_casual"] = data["member_casual"].str.lower().replace({
    "subscriber": "member",
    "customer": "casual"
})

### Step 10: Derived columns

In [30]:
data["ride_length"] = (data["ended_at"] - data["started_at"]).dt.total_seconds() / 60
data["day_of_week"] = data["started_at"].dt.dayofweek

### Step 11: Outlier treatment

In [31]:
data = data[data["ride_length"] > 0]
data = data[data["ride_length"] <= 1440]

### Step 12: Station name cleanup

In [32]:
if "start_station_name" in data.columns:
    data["start_station_name"] = data["start_station_name"].str.strip().str.title()
if "end_station_name" in data.columns:
    data["end_station_name"] = data["end_station_name"].str.strip().str.title()


### Step 13: Final verification

In [33]:
print("Row count:", len(data))
print("Mean ride length (min):", data["ride_length"].mean())
print("Max ride length (min):", data["ride_length"].max())
print("Min ride length (min):", data["ride_length"].min())
print("Rider types:", data["member_casual"].unique())

Row count: 5547380
Mean ride length (min): 14.534683199237001
Max ride length (min): 1439.97595
Min ride length (min): 0.0007666666666666667
Rider types: ['member' 'casual']


### Save cleaned dataset

In [34]:
data.to_csv(os.path.join(folder_path, "cleaned_cyclistic_data.csv"), index=False)